# Lab 01 - Linear Regression with Gradient Descent
**Idea:** fit a line `y = θ0 + θ1·x`. Start with θ = [0, 0], measure the error (cost), and keep nudging θ downhill until the error stops shrinking.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.seterr(all='ignore')   # a too-big learning rate blows up to inf/nan; don't spam warnings

## 1. The functions
- `load_data` - read csv
- `process_data` - (optional) scale x, then add the dummy feature x0 = 1
- `compute_cost` - how wrong the line is
- `gradient_descent` - the learning loop
- `train` - start from θ = 0 and call gradient_descent
- `evaluate` - MSE and R²

In [ ]:
def load_data(path):
    """Read csv -> x (feature), y (target). Last column is y."""
    df = pd.read_csv(path, header=None)
    df = df.apply(pd.to_numeric, errors='coerce').dropna()   # a text header row becomes NaN and is dropped
    return df.iloc[:, 0].values, df.iloc[:, -1].values


def process_data(x, scale=False):
    """Optionally scale x, then add the dummy feature x0 = 1."""
    mean, std = 0, 1                        # default: no scaling
    if scale:
        mean, std = x.mean(), x.std()
    x_scaled = (x - mean) / std
    X = np.c_[np.ones(len(x)), x_scaled]    # column of 1s (x0) + column of x
    return X, mean, std


def compute_cost(X, y, theta):
    m = len(y)
    errors = X @ theta - y                  # predictions - actual values
    return (errors @ errors) / (2 * m)      # J = 1/(2m) * sum(errors^2)


def gradient_descent(X, y, theta, alpha, iters):
    m = len(y)
    costs = []
    for i in range(iters):
        gradient = X.T @ (X @ theta - y) / m    # slope of the cost
        theta = theta - alpha * gradient        # step downhill
        costs.append(compute_cost(X, y, theta))
    return theta, costs


def train(X, y, alpha, iters):
    theta = np.zeros(X.shape[1])            # start from [0, 0]
    return gradient_descent(X, y, theta, alpha, iters)


def evaluate(X, y, theta):
    pred = X @ theta
    mse = np.mean((y - pred) ** 2)
    r2 = 1 - np.sum((y - pred) ** 2) / np.sum((y - y.mean()) ** 2)
    return round(float(mse), 4), round(float(r2), 4)

## 2. Synthetic data: y = 3 + 5x + noise

In [ ]:
np.random.seed(42)
x = np.arange(1, 101)                       # x = 1, 2, ..., 100
y = 3 + 5 * x + np.random.randn(100)        # + standard gaussian noise

plt.scatter(x, y)
plt.xlabel('x'); plt.ylabel('y'); plt.title('Synthetic data')
plt.show()

pd.DataFrame({'x': x, 'y': y}).to_csv('lab01_data.csv', index=False)
x, y = load_data('lab01_data.csv')          # load it back

## 3. The full experiment in one function
1. **Without scaling** - try several learning rates, pick a working one, train
2. **With scaling** - train again
3. Plot the cost curves and the fitted line

In [ ]:
def run_lab(x, y, alpha_raw, iters_raw=5000, alpha_scaled=0.1, iters_scaled=500):

    # ---------- A) WITHOUT feature scaling ----------
    X, mean, std = process_data(x, scale=False)

    print('Learning rate sweep (no scaling, 1000 iterations):')
    for a in [0.1, 0.01, 0.001, 0.0001, 0.00001]:
        _, c = train(X, y, a, 1000)
        print(f'   alpha = {a}: final cost = {c[-1]:.4f}')     # inf/nan = diverged

    theta, costs = train(X, y, alpha_raw, iters_raw)
    print(f'\nNo scaling (alpha={alpha_raw}): theta = {theta}')
    print('MSE, R2 =', evaluate(X, y, theta))
    plt.plot(costs); plt.xlabel('Iteration'); plt.ylabel('Cost')
    plt.title('Training error - no scaling'); plt.show()

    # ---------- B) WITH feature scaling ----------
    Xs, mean, std = process_data(x, scale=True)
    theta_s, costs_s = train(Xs, y, alpha_scaled, iters_scaled)

    # convert theta back to the original x units
    theta_orig = np.array([theta_s[0] - theta_s[1] * mean / std, theta_s[1] / std])
    print(f'\nWith scaling (alpha={alpha_scaled}): theta = {theta_s}')
    print('In original units: theta =', theta_orig)
    print('MSE, R2 =', evaluate(Xs, y, theta_s))
    plt.plot(costs_s); plt.xlabel('Iteration'); plt.ylabel('Cost')
    plt.title('Training error - with scaling'); plt.show()

    # ---------- C) Data + learned line ----------
    plt.scatter(x, y, label='data')
    plt.plot(x, Xs @ theta_s, 'r', label='learned line')
    plt.xlabel('x'); plt.ylabel('y'); plt.legend()
    plt.title('Data with regression line'); plt.show()

In [ ]:
run_lab(x, y, alpha_raw=0.0001)

## 4. Real data (`data_01.csv`)
Upload the file, then run the same experiment. Change `alpha_raw` to the best value from the sweep output.

In [ ]:
from google.colab import files
files.upload()          # choose data_01.csv

In [ ]:
x_real, y_real = load_data('data_01.csv')

plt.scatter(x_real, y_real)
plt.xlabel('x'); plt.ylabel('y'); plt.title('Real data')
plt.show()

run_lab(x_real, y_real, alpha_raw=0.001)